# Expérimentation de la méthode d'évaluation "Retrieval"

## Importation des bibliothèques

In [141]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import pytrec_eval
import json

## Chargement du corpus et du modèle

In [2]:
newsgroups = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))
docs = newsgroups['data']

topic_model = BERTopic(calculate_probabilities=True)
topics, probs = topic_model.fit_transform(docs)

## Retrieval

In [47]:
topic_model.get_document_info(docs)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,\n\nI am sure some bashers of Pens fans are pr...,0,0_game_team_games_he,"[game, team, games, he, players, season, hocke...","[Apparently, Part 2 (defensemen numbered 2 thr...",game - team - games - he - players - season - ...,0.206327,False
1,My brother is in the market for a high-perform...,-1,-1_to_the_and_of,"[to, the, and, of, is, it, in, for, you, that]",[NOTE - local tx groups trimmed out of Newsgro...,to - the - and - of - is - it - in - for - you...,0.414885,False
2,\n\n\n\n\tFinally you said what you dream abou...,-1,-1_to_the_and_of,"[to, the, and, of, is, it, in, for, you, that]",[NOTE - local tx groups trimmed out of Newsgro...,to - the - and - of - is - it - in - for - you...,0.567377,False
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,45,45_scsi_scsi2_scsi1_ide,"[scsi, scsi2, scsi1, ide, asynchronous, contro...",[wlsmith@valve.heart.rri.uwo.ca (Wayne Smith) ...,scsi - scsi2 - scsi1 - ide - asynchronous - co...,0.216354,False
4,1) I have an old Jasmine drive which I cann...,83,83_tape_backup_tapes_drive,"[tape, backup, tapes, drive, munroe, wangdat, ...",[hello all- i have a problem with my micro sol...,tape - backup - tapes - drive - munroe - wangd...,0.188710,False
...,...,...,...,...,...,...,...,...
18841,DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...,-1,-1_to_the_and_of,"[to, the, and, of, is, it, in, for, you, that]",[NOTE - local tx groups trimmed out of Newsgro...,to - the - and - of - is - it - in - for - you...,0.667745,False
18842,\nNot in isolated ground recepticles (usually ...,182,182_ground_grounding_conductor_neutral,"[ground, grounding, conductor, neutral, wire, ...","[\nNot according to the NEC nor the CEC, as ex...",ground - grounding - conductor - neutral - wir...,1.000000,False
18843,I just installed a DX2-66 CPU in a clone mothe...,88,88_fan_cpu_heat_sink,"[fan, cpu, heat, sink, fans, cooling, chip, co...","[N(P>Just got a 66MHz 486DX2 system, and am co...",fan - cpu - heat - sink - fans - cooling - chi...,0.332336,False
18844,\nWouldn't this require a hyper-sphere. In 3-...,18,18_den_polygon_points_algorithm,"[den, polygon, points, algorithm, xxxx, plane,...","[\nSorry!! :-)\n\nCall the four points A, B, C...",den - polygon - points - algorithm - xxxx - pl...,0.270765,False


In [86]:
def getWordsTopics(topicsKey :int, model : BERTopic) -> list :
    return [word for word,_ in model.get_topic(topicsKey)]

In [151]:
def prepare_data(topic_model: BERTopic) -> tuple[dict, dict]:
    """
    Prépare le dictionnaire qrel et run pour la bibliothèque pytrec_eval
    topic_model: modèle BERTopic
    return: dictionnaire qrel et run
    """
    
    documentInfos = topic_model.get_document_info(docs=docs)
    topicKeys = topic_model.get_topics().keys()
    qrel = dict()
    predictions = dict()

    for i, topic in enumerate(topicKeys):
        topic_id = str(topic)
        words = getWordsTopics(topic, topic_model)
        qrel[topic_id] = {}
        predictions[topic_id] = {}
        embeddingMotsCles = topic_model.embedding_model.embed_words(words=words)
        for j, doc in tqdm(documentInfos.iterrows()):
            text = doc['Document']
            embeddingDoc = topic_model.embedding_model.embed_documents(document=[text])
            doc_id = str(j)
            predictions[topic_id][doc_id] = float(cosine_similarity(embeddingMotsCles, embeddingDoc)[0][0])
            if topic == doc['Topic']:
                qrel[topic_id][doc_id] = 1
            else:
                qrel[topic_id][doc_id] = 0
    return qrel, predictions

In [152]:
def retrieval(topic_model: BERTopic) -> {}:
    """
    Calcule les métriques de "retrieval" en préparant le dictionnaire qrel et run à l'aide de la bibliothèque pytrec_eval.
    Métriques : 
        - Mean Average Precision (MAP)
        - Normalized Discounted Cumulative Gain (NDCG)
    topic_model: modèle BERTopic
    return: dictionnaire des métriques
    """
    
    qrel, predictions = prepare_data(topic_model=topic_model)

    evaluator = pytrec_eval.RelevanceEvaluator(qrel, {'map', 'ndcg'})
    results = evaluator.evaluate(predictions)
    
    return results

In [153]:
results = retrieval(topic_model=topic_model)

6771it [04:41, 24.04it/s]


KeyboardInterrupt: 